# Adapting minibatch sample size and number of replicates across generations

The idea is that we calculate variances within replicates and within batches. Error in summary statistics due to batches provides the first layer of adaptability for batch size. The variation of a summary statistic due to batch size guesses will affect acceptance rates (Basically via classification). The replicate number then sharpens the summary statistics for specific batches. We could achieve a similar albeit uneven effect by ramping up batch size way beyond sample size, though this wouldn't be principled and we would have to deal with variable sampling error. So, we can decompose the sampling error like this:

$$
\frac{v_{batch}}{kn} + \frac{v_{rep}}{k} = c^2 \epsilon^2 + \frac{v_{batch}}{N}
$$

The adaptive rule becomes...
$$
n = \frac{v_{batch}}{v_{rep} + k(c^2\epsilon^2 + v_{batch})}
$$
$$
k = \frac{v_{batch} + nv_{rep}}{n(c^2\epsilon^2 + v_{batch})}
$$

We calculate $v_{rep}$ similar to how we calculate $v_{batch}$.

In [1]:
#Inference classes
from smc_abc import smc_abc_iterator as abc_iter
import smc_abc_utils as utils
import smc_abc_schemes as schemes

#Simulator classes
import predator_prey
lv_sim = predator_prey.lotka_volterra

#Support
import numpy as np
from matplotlib import pyplot as plt


cwd appended to system path


In [2]:
## Setup and tau leaping example
sample_size = 10
alpha,beta,gamma,delta = 4,0.01,4,1
K = 10000
t_max = 5
dt = 0.01
ic = np.random.uniform(400,600,size=(sample_size,2))

times,data = lv_sim.tau_leaping(ic,t_max,alpha,beta,gamma,delta,dt = dt,K = K)

print(np.sum(np.any(np.any(data == 0,axis=0),axis=1)))

0


In [3]:
#general parameters
num_particles = 300
cores = 4

#Stats function
def batch_corr(a, b):
    """Vectorized correlation between rows of a and b"""
    ma = a - np.mean(a, axis=1, keepdims=True)
    mb = b - np.mean(b, axis=1, keepdims=True)
    
    num = np.sum(ma * mb, axis=1)
    den = np.sqrt(np.sum(ma**2, axis=1) * np.sum(mb**2, axis=1))
    return num / (den + 1e-8)

def stats_func_lv(x, max_lag=5, step_size=10):
    r = x[:, :, 0]
    f = x[:, :, 1]
    
    # 1. Base Moments (Per-particle)
    stats_list = [
        np.log(np.mean(r, axis=1) + 1e-8),
        np.log(np.mean(f, axis=1) + 1e-8),
        np.log(np.var(r, axis=1) + 1),
        batch_corr(r, f) # Cross-correlation
    ]

    # 2. Add multiple Autocorrelation lags
    for i in range(1, max_lag + 1):
        lag = i * step_size # e.g., lags of 10, 20, 30, 40, 50 steps
        stats_list.append(batch_corr(r[:, :-lag], r[:, lag:]))
        stats_list.append(batch_corr(f[:, :-lag], f[:, lag:]))

    return np.column_stack(stats_list)
def stats_func(x,y,sf=stats_func_lv):
    return sf(x),sf(y)


def model(p, Bi, num_reps=1):
    alpha,beta,gamma = p
    Bi_rep = np.repeat(Bi,repeats=num_reps)
    X0 = ic[Bi_rep]
    _, sim = lv_sim.tau_leaping(
        X0,t_max,
        alpha,beta,gamma,delta,
        dt=dt, K=K
    )
    s_sim = stats_func_lv(sim)
    obs = data[Bi_rep]
    s_obs = stats_func_lv(obs)
    return s_sim, s_obs


def stats_func(x,y,sf=stats_func_lv):
    return x,y

prior = utils.uniform_prior([[0.0,10.0],[0.0,0.05],[0.0,10.0]])

s_data = stats_func_lv(data)

init_params = {
    'data': s_data,
    'model': model,
    'stats_func':stats_func,
    'prior':prior,
    'num_particles':num_particles,
    'cores':cores
}

In [4]:
def constant_init(init_params,p=[1]):
    est = abc_iter(**init_params)
    est.batch_size = p[0]
    est.esv = lambda S: np.linalg.trace(est.W_inv @ S)
    est.v_total_est = np.inf
    est.init_sigma = est.esv(est.W_inv)
    def estimate_v_total(est):
        stats_ref_expanded = np.repeat(est.posterior_stats_ref,repeats = est.reps,axis=1)
        delta = est.posterior_stats - stats_ref_expanded
        if delta.ndim <= 2:
            Sigma = np.cov(delta.T)
        else: #The dimensions are (N particles by N_bs batch size by N_s stats)
            cov_list = np.array([np.cov(x.T) for x in delta]) #(N by N_s by N_s)
            Sigma = np.mean(cov_list,axis=0)
        est.Sigma = Sigma
        est.v_total_est = est.esv(Sigma)
    est.estimate_v_total = estimate_v_total
    est.snr = np.inf
    return est
def constant_loop(est,ess_resample=True,ess_prop = 0.5):
    est.generate()
    if ess_resample:
        est.ESS_resample(ess_prop)
    est.estimate_v_total(est)
    v = est.v_total_est
    nt = est.batch_size
    N = est.sample_size
    esq = est.next_alpha_threshold**2
    est.c_est_ = np.sqrt((v/esq)*(1/nt - 1/N))
    est.rhs = v/N + (est.c_est_)**2 * esq
    est.lhs = v/nt
    snr = esq / (v / nt)
    est.snr = snr
    print(f"Estimated v = {v:.3f}. Noise: v/n = {v/nt:.3f}. v/N = {v/N:.3f}. Estimated c = {est.c_est_:.3f}. SNR: eps^2 / (v/n) = {snr:.3f}. Time per sim: {est.step_time / est.num_sims:.3f}")


In [15]:
def fvc_kn_init(args,p=[2,None,1.0,0.5], asymptotic = False):
    '''
    Variance-control adaptive minibatch SMC ABC applied to adapting replicates. For stochastic simulators only. 
    
    Parameters
    ----------
    args : dictionary
        A dictionary of inputs to the SMC ABC class. Requires data, model, stats function, and prior at least.
    p : list
        Hyper parameters for the method: [initial minibatch size, c, lambda, alpha]
    asymptotic : Bool
        If False, uses the finite sample size formula for adapting replicates. 
    '''
    est = constant_init(args)
    est.batch_size = p[0]
    est.c = p[1]
    est.lambda_ = p[2] #since "lambda" is a special word, add an undercore
    l = p[2]
    alpha = p[3]
    est.reps = 2
    num_reps = est.reps
    N = est.sample_size
    W_inv = est.W_inv
    est.esv = lambda S: np.linalg.trace(W_inv @ S)
    est.base_model = est.model #necessary for handling recussion
    est.model = lambda p,Bi : est.base_model(p,Bi,int(num_reps))
    def update_mbs():
        stats_ref_expanded = np.repeat(est.posterior_stats_ref,repeats = est.reps,axis=1)
        delta = est.posterior_stats - stats_ref_expanded
        n_particles = delta.shape[0]
        n_total_rows = delta.shape[1]
        n_stats = delta.shape[2]
        n_batches = est.__batch_size_round__
        n_reps = int(est.reps)
        if delta.ndim <= 2: #This part shouldn't apply anymore.... Remove it....
            Sigma = np.cov(delta.T)
        else: #The dimensions are (N particles by N_bs batch size by N_s stats)
            #The idea here is to calculate ESV within replicates, Average over replicates, then continue the computation. 
            #We would obtain within-replicate variance and within-batch variance. 
            delta_4d = delta.reshape(n_particles, n_batches, n_reps, n_stats)
            replicate_means = np.mean(delta_4d, axis=2)
            obs_sigma = replicate_means.reshape(-1, n_stats)
            Sigma = np.cov(obs_sigma.T)

            groups_for_rep = delta_4d.reshape(-1,n_reps,n_stats)
            cov_list_rep = [np.atleast_2d(np.cov(group.T)) for group in groups_for_rep]
            Sigma_rep = np.mean(cov_list_rep,axis=0)
            # cov_list_rep = np.array([np.cov(x.T) for x in delta.reshape(est.batch_size,est.reps,-1)])
            # Sigma_rep = np.mean(cov_list_rep,axis=0)
            # delta_rep_ave = np.mean(delta.reshape(est.batch_size,est.reps,-1),axis=1)
            # #Assuming we have successive batches in replicates, we would have
            # cov_list = np.array([np.cov(x.T) for x in delta_rep_ave]) #(N by N_s by N_s)
            # Sigma = np.mean(cov_list,axis = 0) #(N_s,N_s)
        est.Sigma = Sigma
        v_total = est.esv(Sigma)
        # v_total = np.maximum(v_total,1e-12)
        v_total_rep = est.esv(Sigma_rep)
        v_total_rep = np.maximum(v_total_rep,1e-12)
        # Exponential Moving Average update v_total
        est.v_total = (1-l) * est.v_total + l * v_total if est.generation > 1 else v_total
        est.v_total_rep = (1-l) * est.v_total_rep + l * v_total_rep if est.generation > 1 else v_total_rep
        est.c_est = np.sqrt((est.v_total*(N - est.batch_size))/(est.batch_size*N*est.current_alpha_threshold**2))

        #automatic c selection
        if est.c is None:
            if asymptotic:
                est.c = np.sqrt(est.v_total/(est.reps * est.batch_size * est.alpha_threshold**2))
            else:
                est.c = np.sqrt((est.v_total*(N - est.reps*est.batch_size))/(est.reps*est.batch_size*N*est.alpha_threshold**2))
            new_hyperparams = [est.batch_size,est.c,est.lambda_]
            print(f"Automatic c selection: c = {est.c}")
            print(f"New hyperparameter set: {new_hyperparams}")
        
        n = est.batch_size
        k = est.reps
        # batch_size = v_total / (-v_total_rep + k*(est.c**2 * est.alpha_threshold**2 + v_total/N))
        # num_reps = (v_total + n * v_total_rep) / (n * (est.c**2 * est.alpha_threshold**2 + v_total/N))
        batch_size = np.sqrt(v_total/v_total_rep)
        num_reps = (v_total_rep + v_total/batch_size)/(est.c**2 * est.alpha_threshold**2 + v_total / est.sample_size)

        print(batch_size,num_reps)

        est.batch_size = batch_size
        est.reps = np.maximum(num_reps,2)
        num_reps = est.reps
        est.model = lambda p,Bi : est.base_model(p,Bi,int(num_reps))
        print(f"Variance estimate: {est.v_total/N + (est.c*est.alpha_threshold)**2:.3f}. Within-batch variance: {est.v_total:.3f}. Within-replicate variance {v_total_rep:.3f}. Estimated K {k:.3f}.C estimate: {est.c_est:.3e}. batch size: {est.batch_size}. reps: {num_reps:.1f}")
    est.update_mbs = update_mbs
    return est

def fvc_loop(est,ess_resample=True,ess_prop = 0.5):
    constant_loop(est,ess_resample,ess_prop)
    est.update_mbs()

In [16]:
est = fvc_kn_init(init_params,p = [10,None,1,1.0],asymptotic=True)

for i in range(15):
    fvc_loop(est)

Generation: 1. Batch Size: 10. Acceptance Rate: 1.000. Step time: 6.357. Total time 6.357. post-filter threshold: 214.3845. ESS: 300.00. Log HDPR vol: 1.105
Estimated v = 16304.613. Noise: v/n = 1630.461. v/N = 1630.461. Estimated c = 0.000. SNR: eps^2 / (v/n) = 28.189. Time per sim: 0.011
Automatic c selection: c = 0.15120623323238414
New hyperparameter set: [np.int64(10), np.float64(0.15120623323238414), 1]
1.1571133974732752 10.740632894911101
Variance estimate: 3152.443. Within-batch variance: 21016.286. Within-replicate variance 15696.547. Estimated K 2.000.C estimate: 0.000e+00. batch size: 2.0. reps: 10.7
Generation: 2. Batch Size: 2. Acceptance Rate: 0.634. Step time: 8.051. Total time 14.408. post-filter threshold: 86.8638. ESS: 289.86. Log HDPR vol: 0.384
Estimated v = 2456.659. Noise: v/n = 1228.329. v/N = 245.666. Estimated c = 0.361. SNR: eps^2 / (v/n) = 6.143. Time per sim: 0.009
1.1545463617204084 10.588040729813065
Variance estimate: 500.123. Within-batch variance: 3276

KeyboardInterrupt: 

### The one below really works for some reason....

In [ ]:
#general parameters
num_particles = 300
cores = 4


#Stats function
def batch_corr(a, b):
    """Vectorized correlation between rows of a and b"""
    ma = a - np.mean(a, axis=1, keepdims=True)
    mb = b - np.mean(b, axis=1, keepdims=True)
    
    num = np.sum(ma * mb, axis=1)
    den = np.sqrt(np.sum(ma**2, axis=1) * np.sum(mb**2, axis=1))
    return num / (den + 1e-8)

def stats_func_lv(x, max_lag=5, step_size=10):
    r = x[:, :, 0]
    f = x[:, :, 1]
    
    # 1. Base Moments (Per-particle)
    stats_list = [
        np.log(np.mean(r, axis=1) + 1e-8),
        np.log(np.mean(f, axis=1) + 1e-8),
        np.log(np.var(r, axis=1) + 1),
        batch_corr(r, f) # Cross-correlation
    ]

    # 2. Add multiple Autocorrelation lags
    for i in range(1, max_lag + 1):
        lag = i * step_size # e.g., lags of 10, 20, 30, 40, 50 steps
        stats_list.append(batch_corr(r[:, :-lag], r[:, lag:]))
        stats_list.append(batch_corr(f[:, :-lag], f[:, lag:]))

    return np.column_stack(stats_list)
def stats_func(x,y,sf=stats_func_lv):
    return sf(x),sf(y)


def model(p, Bi, num_reps=1):
    alpha,beta,gamma = p
    Bi_rep = np.repeat(Bi,repeats=num_reps)
    X0 = ic[Bi_rep]
    _, sim = lv_sim.tau_leaping(
        X0,t_max,
        alpha,beta,gamma,delta,
        dt=dt, K=K
    )
    s_sim = stats_func_lv(sim)
    obs = data[Bi_rep]
    s_obs = stats_func_lv(obs)
    return s_sim, s_obs


def stats_func(x,y,sf=stats_func_lv):
    return x,y

prior = utils.uniform_prior([[0.0,10.0],[0.0,0.05],[0.0,10.0]])

s_data = stats_func_lv(data)

init_params = {
    'data': s_data,
    'model': model,
    'stats_func':stats_func,
    'prior':prior,
    'num_particles':num_particles,
    'cores':cores
}

In [ ]:
est = schemes.fvc_rep_init(init_params, p = [sample_size, None, 1.0], asymptotic=True)

for i in range(15):
    schemes.fvc_loop(est)

In [ ]:
est = schemes.fvc_kn_init(init_params, p = [sample_size, None, 1.0,0.5,4], asymptotic=True)

for i in range(15):
    schemes.fvc_loop(est)